# 文件读写与异常处理

## 1. 为什么需要持久化？—— RAM 是易失的

计算机的内存（RAM）就像一块**白板**：程序运行时可以在上面快速书写和擦除，
但**一旦断电，所有内容都会消失**。硬盘（或 SSD）则像一本**笔记本**——
读写速度慢一些，但关机后内容依然存在。

### 生活中的类比

| 内存 (RAM) | 硬盘 (持久存储) |
|-----------|----------------|
| 大脑的短期记忆 | 纸质笔记本 |
| 关电脑 = 失忆 | 关电脑 = 合上笔记本 |
| 极快但易失 | 较慢但持久 |

### OS 层面：从进程到文件的旅程

当你在 Python 中调用 `open("file.txt", "r")` 时，操作系统层面发生了这些事：

1. **系统调用**：Python 的 `open()` 会触发内核的 `open()` 系统调用，CPU 从用户态切换到内核态
2. **文件描述符（File Descriptor）**：内核返回一个**非负整数**（如 3、4、5），作为该文件的句柄。进程的文件描述符表存在 PCB（进程控制块）中，0=stdin, 1=stdout, 2=stderr，新打开的 fd 通常是 3
3. **VFS（虚拟文件系统）**：Linux 通过 VFS 层屏蔽了 ext4、NTFS、NFS 等不同文件系统的差异，让 `open()`、`read()` 等接口统一
4. **内核缓冲区（Page Cache）**：`read()` 并不直接读磁盘，而是先在内核的 Page Cache 中查找。如果命中，直接返回；否则触发磁盘 I/O，将数据读入 Page Cache 再返回
5. **写缓冲**：`write()` 调用只是把数据写入内核缓冲区，CPU 立即返回。内核后台通过 **pdflush**（Linux）或 **dirty page** 机制异步写入磁盘

```
Python 进程 → open() 系统调用 → VFS → 具体文件系统 (ext4/NTFS)
                                    ↓
                              Page Cache (内核缓冲区)
                                    ↓
                              磁盘/SSD 硬件
```

### 常见误区

- **"数据在变量里存着就行"** —— 程序退出后变量全部销毁
- **"写到内存就够了，不需要存文件"** —— 重启后一切归零
- **"文件操作很简单，不用学"** —— 实际上文件编码、模式、异常处理都是坑
- **"写入文件 = 立即写到磁盘"** —— 不是！`f.write()` 只是写到内核缓冲区，系统崩溃仍可能丢数据。需要 `os.fsync(f.fileno())` 强制刷盘

### Python 中文件操作的三层抽象

| 层次 | 接口 | 说明 |
|------|------|------|
| 应用层 | `open()` / `read()` / `write()` | Python 封装，最常用 |
| IO 层 | `io.FileIO` / `io.BufferedReader` | 字节级别的读写缓冲 |
| 系统调用 | `os.open()` / `os.read()` / `os.write()` | 直接调用 POSIX 系统调用 |
| 内核 | VFS → Page Cache → 块设备层 | 操作系统内核内部实现 |

Python 提供了强大而简洁的文件操作 API，配合 `with` 语句和异常处理，
可以安全、高效地管理数据持久化。理解底层原理，能帮你写出更健壮的代码。

## 2. 文件打开、读取与写入 —— `with` 语句

### 为什么 `with` 比手动 `close()` 更好？

传统写法需要手动调用 `f.close()`，但很容易忘记，或者在发生异常时跳过关闭，
导致**文件句柄泄漏**（就像借了书不还，图书馆的书越来越少）。

```python
# 不推荐的写法
f = open("data.txt", "w")
f.write("hello")
f.close()  # 如果 write 报错，close 永远不会执行
```

`with` 语句会在代码块结束时（无论是否发生异常）**自动关闭文件**，
就像 `try / finally` 的语法糖。

#### 深入理解上下文管理器协议

`with` 语句的本质是调用**上下文管理器协议**，包含两个特殊方法：

```python
class File:
    def __enter__(self):
        # 进入 with 块时调用，返回值赋值给 as 后面的变量
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        # 离开 with 块时调用，无论是否发生异常
        # exc_type: 异常类型（无异常时为 None）
        # exc_val: 异常实例（无异常时为 None）
        # exc_tb: 异常回溯对象（无异常时为 None）
        self.close()
        # 返回 False 或 None：异常继续传播
        # 返回 True：异常被吞没（通常不推荐）
        return False
```

所以 `with open(...) as f` 等价于：

```python
f = open(...).__enter__()       # with 开始时调用 __enter__
try:
    ...  # 你的代码
finally:
    f.__exit__(None, None, None)  # with 结束时自动调用 __exit__
```

这也是为什么你可以自定义任何类实现上下文管理器——不只是文件对象！
比如数据库连接、锁、网络 socket 都可以用 `with` 管理。

### OS 层面：close() 时发生了什么？

调用 `f.close()`（或 `__exit__` 自动调用）时，操作系统层面会：

1. **刷新用户态缓冲区**：Python 的 `io` 层（`BufferedWriter`）将内部缓冲区余下的字节刷入内核
2. **刷新内核缓冲区**：调用 `fsync()` 系统调用，强制将 Page Cache 中的脏页写入磁盘（默认 `close()` 也会刷盘，但系统崩溃时可能丢数据）
3. **释放文件描述符**：内核从进程的 fd 表中移除该条目，fd 可被后续 `open()` 重用
4. **降低 inode 引用计数**：如果这是最后一个 fd，inode 的引用计数归零

**不 close 的后果**：进程的 fd 表默认总大小有限（Linux 可用 `ulimit -n` 查看，通常 1024），不关闭文件会导致**文件描述符泄漏**，最终无法打开新文件。

### CPU 时间：用户态 vs 内核态

```
f.write("hello")          → 纯用户态操作（Python 字节码执行 + str 编码为 bytes）
内核缓冲区写入           → 切换到内核态（系统调用 write()）
pdflush 异步刷盘         → 内核态（后台线程，不阻塞你的代码）
f.close() → fsync()      → 内核态（同步等待刷盘完成，可能较慢）
```

### 常见误区

- **忘记模式参数** —— `open("file")` 默认是 `"r"`（只读），尝试写入会抛 `io.UnsupportedOperation`
- **中文编码问题** —— 不带 `encoding="utf-8"` 可能在 Windows 上使用系统默认编码（如 gbk）导致乱码
- **认为 with 块外还能读写** —— 离开缩进块后文件已关闭，任何操作都会抛 `ValueError: I/O operation on closed file`
- **`f.flush()` 不等于 `fsync()`** —— `flush()` 只把 Python 用户态缓冲区的数据刷入内核缓冲区，`os.fsync(f.fileno())` 才保证数据写入磁盘
- **`close()` 可能抛出异常** —— 如果 `flush()` 或 `fsync()` 失败（如磁盘满），`close()` 会抛出 `IOError`

In [ ]:
# === 文件读写基础 ===

# ---- 写入文件 ----
# open() 是 Python 内置函数，向操作系统发起系统调用请求打开一个文件
# 参数 "/tmp/demo_write.txt"：文件路径（绝对路径，指向 /tmp 目录）
# 参数 "w"：write 模式——如果文件已存在则清空覆盖，如果不存在则创建新文件
# 参数 encoding="utf-8"：指定文本编码为 UTF-8，避免 Windows 默认 gbk 导致乱码
# with 语句会调用文件对象的 __enter__ 和 __exit__ 方法，保证离开缩进块时自动 f.close()
with open("/tmp/demo_write.txt", "w", encoding="utf-8") as f:
    # f.write() 将字符串写入文件缓冲区，数据先暂存在内核缓冲区，f.close() 时刷入磁盘
    f.write("Hello, 世界！\n")
    # 再次调用 write()，内容追加到文件指针当前位置（上一行末尾之后）
    f.write("这是第二行。\n")
# 离开 with 缩进块 → 自动调用 f.close() → 触发操作系统 fsync() 将缓冲区数据真正写入磁盘

print("写入完成。")

# ---- 读取文件 ----
# "r"：read 模式——文件必须已存在，否则抛出 FileNotFoundError
with open("/tmp/demo_write.txt", "r", encoding="utf-8") as f:
    # f.read() 一次性读取整个文件内容到内存，返回字符串
    # 对于大文件不推荐这样用，会占用大量内存
    content = f.read()
# 离开 with 块后文件自动关闭

print("读取内容：")
print(content)

# ---- 验证 with 块外文件已关闭 ----
# 在 with 块外部试图访问文件对象，应该抛出异常
try:
    f.read()  # 文件已关闭，f.read() 会触发 ValueError
except ValueError as e:
    print(f"块外访问报错：{e}")

## 3. 文件模式（File Modes）

文件模式决定了**打开文件的方式**，就像你对待一本笔记本的态度：

| 模式 | 含义 | 文件指针 | 如果文件已存在 | 如果文件不存在 | 典型场景 |
|------|------|---------|---------------|---------------|---------|
| `"r"` | 只读 | 开头 | 正常读取 | 报错 `FileNotFoundError` | 读取配置文件 |
| `"w"` | 只写 | 开头 | **清空覆盖** | 创建新文件 | 写入日志、导出数据 |
| `"a"` | 追加 | **末尾** | 在末尾追加 | 创建新文件 | 记录日志、追加数据 |
| `"x"` | 排他创建 | 开头 | **报错** `FileExistsError` | 创建新文件 | 防止覆盖重要文件 |
| `"r+"` | 读写 | 开头 | 正常读写 | 报错 | 需要同时读写 |
| `"w+"` | 读写 | 开头 | 清空覆盖 | 创建 | 读写新文件 |
| `"a+"` | 读写追加 | 末尾 | 读写追加 | 创建 | 读+追加写 |

### 模式对比：真实场景的陷阱

**`"w"` 的破坏性**：这是最危险的模式。`"w"` 会在打开文件的**瞬间**就截断文件（truncate），而不是在第一次 `write()` 时。即使你只打开什么都没写，原文件内容也已经丢失了。

```python
# 危险！文件在 with 块开始前就被清空了，即使 open 后就报错也一样
with open("important_data.csv", "w", encoding="utf-8") as f:
    # 此时文件内容已经没了！
    raise RuntimeError("出错了！")
# 即使异常退出，文件也已经变成了空文件 ❌
```

**`"a"` 的 seek 陷阱**：追加模式下，每次 `write()` 都会将数据写到文件末尾，不受 `seek()` 影响。但 `"a+"` 模式下可以 `seek()` 到前面用 `read()` 读取。

**`"x"` 的原子性**：`"x"` 模式底层使用 `O_EXCL` 标志（Linux 系统调用），在多进程场景下可以原子性地检查并创建文件，避免竞态条件。

### 二进制模式 vs 文本模式

在模式后加 `b`（如 `"rb"`、`"wb"`）表示以**二进制模式**打开：

| 特性 | 文本模式 (`"r"`, `"w"`, `"a"`) | 二进制模式 (`"rb"`, `"wb"`, `"ab"`) |
|------|------|------|
| 数据接口 | 读写 `str` 对象 | 读写 `bytes` 对象 |
| 编码转换 | 自动编码/解码（按 `encoding=` 参数） | 无编码转换，原始字节 |
| 换行符转换 (Unix) | 无变化 | 无变化 |
| 换行符转换 (Windows) | `\n` ↔ `\r\n` 自动转换 | **无转换** |
| 适用场景 | 文本文件 (`.txt`, `.csv`, `.json`, `.py`) | 二进制文件 (`.png`, `.mp3`, `.zip`, `.exe`) |

**换行符转换细节**：在 Windows 上，文本模式读文件时，`\r\n` 被自动转换为 `\n`；写文件时，`\n` 被自动转换为 `\r\n`。在 Linux/macOS 上则无此转换。这就是为什么 `.py` 脚本在 Windows 上使用 `"rb"` 读取会得到 `\r\n` 的原因。

**BOM（Byte Order Mark）**：UTF-16 文件通常以 BOM（`\xff\xfe` 或 `\xfe\xff`）开头，标识字节序。UTF-8 理论上不需要 BOM，但 Windows 上的记事本会在 UTF-8 文件开头添加 `\xef\xbb\xbf`。Python 的 `utf-8-sig` 编码可以自动处理 UTF-8 BOM。

### OS 层面文件打开标志

Python 模式到 Linux 系统调用标志的映射（供有兴趣的读者参考）：

| Python 模式 | Linux open() 标志 | 说明 |
|-------------|-------------------|------|
| `"r"` | `O_RDONLY` | 只读打开 |
| `"w"` | `O_WRONLY \| O_CREAT \| O_TRUNC` | 写入，创建，截断 |
| `"a"` | `O_WRONLY \| O_CREAT \| O_APPEND` | 写入，创建，追加 |
| `"x"` | `O_WRONLY \| O_CREAT \| O_EXCL` | 写入，创建，排他 |

### 常见误区

- **`"w"` 会直接清空文件** —— 这是一个破坏性操作，没有撤销功能
- **认为 `"a"` 模式下可以随机读写** —— 追加模式下 `seek()` 定位只影响读取，不影响写入位置
- **混淆文本模式和二进制模式** —— 读图片用 `"r"` 会导致 `UnicodeDecodeError`
- **Windows 上二进制模式写 CSV 不需要 `newline=""`** —— 因为二进制模式已经禁用了换行符转换
- **`"r+"` 不会创建文件** —— 这是最常见的误解！`"r+"` 要求文件必须已存在，不然会报错

In [ ]:
# === 文件模式演示 ===
import os  # os.path.exists(), os.remove() 等文件和路径操作函数

print("=== 模式 'w'：写入（覆盖） ===")
# "w" = write 模式：以写入方式打开，文件指针置于开头
# 如果文件已存在→**清空所有内容**；如果不存在→创建新文件
# 这是一个破坏性操作，使用时务必谨慎！
with open("/tmp/mode_test.txt", "w", encoding="utf-8") as f:
    f.write("第一版内容\n")
print("文件已创建。")

# 再次用 "w" 模式打开同一文件——原来的内容会被完全清空
with open("/tmp/mode_test.txt", "w", encoding="utf-8") as f:
    f.write("被覆盖了！\n")
print("再次写入（覆盖）完成。")

# "r" = read 模式：只读方式打开，文件指针置于开头
# 如果文件不存在→抛出 FileNotFoundError
with open("/tmp/mode_test.txt", "r", encoding="utf-8") as f:
    print(f"读取结果：{f.read()!r}")  # !r 用 repr() 输出，能看到换行符 \n

print("\n=== 模式 'a'：追加 ===")
# "a" = append 模式：以写入方式打开，文件指针置于**文件末尾**
# 写入的内容会追加到现有内容之后，不会覆盖已有数据
# 如果文件不存在→创建新文件
with open("/tmp/mode_test.txt", "a", encoding="utf-8") as f:
    f.write("追加的内容\n")

with open("/tmp/mode_test.txt", "r", encoding="utf-8") as f:
    print(f"追加后读取：{f.read()!r}")

print("\n=== 模式 'x'：排他创建 ===")
# "x" = exclusive creation 模式：以写入方式打开，文件指针置于开头
# 如果文件已存在→抛出 FileExistsError（这正是保护机制！）
# 如果文件不存在→创建新文件
# 适合需要确保不覆盖已有文件的场景（如临时锁文件、配置文件初始化）
try:
    with open("/tmp/exclusive_test.txt", "x", encoding="utf-8") as f:
        f.write("排他创建")
except FileExistsError:
    print("文件已存在，x 模式报错！（这正是我们想要的保护机制）")

print("\n=== 二进制模式 'wb' ===")
# "wb" = write + binary 模式：以二进制写入方式打开
# 与文本模式的区别：不进行编码转换（不处理 str↔bytes），不转换换行符
# 读写图片、音频、视频等非文本文件时必须用二进制模式
with open("/tmp/binary_test.bin", "wb") as f:
    f.write(b"\\x00\\x01\\x02")  # b 前缀表示 bytes 字面量，直接写入原始字节

# "rb" = read + binary 模式：以二进制方式读取
# f.read() 返回 bytes 对象而不是 str
with open("/tmp/binary_test.bin", "rb") as f:
    data = f.read()
print(f"二进制数据：{data}")
print(f"长度：{len(data)} 字节")

# 清理临时文件
# os.path.exists() 检查文件是否存在，避免 os.remove() 抛出 FileNotFoundError
for f in ["/tmp/mode_test.txt", "/tmp/exclusive_test.txt", "/tmp/binary_test.bin"]:
    if os.path.exists(f):
        os.remove(f)  # os.remove() 发起系统调用删除文件
print("\n临时文件已清理。")

## 4. 逐行读取大文件

当文件**非常大**（如几 GB 的日志文件）时，一次性 `f.read()` 会占用巨量内存，
可能导致程序崩溃甚至系统卡顿。

### 核心理念：不要一次性全部读入

就像吃自助餐：你不会把整桌菜一次性塞进嘴里，而是一口一口地吃。
同样，大文件应该**逐行处理**，每次只占用一行数据的内存。

### OS 层面：缓冲区是如何工作的

当你用 `for line in f` 逐行读取时，Python 内部并非真的每次读取一个字节找换行符，
而是使用**双重缓冲机制**：

```
磁盘 → 内核缓冲区 (Page Cache，默认 4KB) → 用户态缓冲区 (io.BufferedReader，默认 8192 字节) → 你的代码
```

具体过程：
1. Python 的 `BufferedReader` 一次向内核读取 8192 字节（8KB）到用户态缓冲区
2. `for line in f` 在用户态缓冲区中查找 `\n`，按行 yield 给调用者
3. 缓冲区消耗完毕后，再次触发系统调用读取下一块 8KB
4. **不管文件有多大，永远只有 8KB 数据在内核和用户态之间传输**——这就是内存高效的原因

可以通过 `open("file", "r", buffering=65536)` 调整缓冲区大小，但默认 8192 对大多数场景已经足够。

### 三种逐行读取方式

| 方法 | 代码 | 特点 |
|------|------|------|
| `for line in f` | 迭代器方式 | **最推荐**，内存高效，语法简洁，底层自动缓冲 |
| `f.readline()` | 手动读取 | 需要自己控制循环，容易写出死循环 |
| `f.readlines()` | 一次读取所有行 | **不推荐**，仍然全部加载到内存 |

### 深入：为什么 `for line in f` 是迭代器？

文件对象实现了 `__iter__()` 和 `__next__()` 方法，因此是**可迭代对象**：

```python
with open("file.txt") as f:
    it = iter(f)       # f.__iter__() 返回迭代器
    line1 = next(it)   # f.__next__() 读取一行
    line2 = next(it)   # 继续读取下一行
```

`for` 循环底层就是不断调用 `next(it)` 直到 `StopIteration` 被抛出。这意味着：
- 每次只保留**一行**数据在内存
- 读取速度受限于磁盘 I/O，而不是内存大小
- 适合配合 `enumerate()` 获取行号

### 常见误区

- **`readlines()` 和逐行读取混淆** —— `readlines()` 把所有行加载到列表，大文件别用
- **忘记去除换行符** —— `line.strip()` 可以同时去除两端空白和 `\n`；如果只需去行末换行符，用 `line.rstrip('\n')` 更精确
- **二进制模式下逐行读取** —— `for line in f` 在二进制模式下仍然可以工作，但分割的是 `\n` 字节（`b'\n'`），返回的是 `bytes` 对象
- **"逐行"不一定是"按行"** —— 如果文件中某一行长达 500MB（如单行 JSON），`for line in f` 仍然会一次性读入 500MB，因为它是按**换行符**分割的
- **`readline()` 的空字符串判断** —— 注意空行返回 `'\n'`（长度为 1，不是空字符串），只有真正到文件末尾才返回 `''`

In [ ]:
# === 逐行读取演示 ===

# ---- 创建一个包含 1000 行的测试文件 ----
# 先用 "w" 模式生成一个测试文件，模拟大文件的场景
with open("/tmp/big_file.txt", "w", encoding="utf-8") as f:
    # range(1, 1001) 生成 1 到 1000 的整数序列
    for i in range(1, 1001):
        # f.write() 逐行写入，:04d 表示数字占 4 位，不足补零（如 0001）
        f.write(f"第{i:04d}行：这是示例数据行。\n")

print("已创建包含 1000 行的测试文件。")

# ---- 方式一（推荐）：for line in f — 迭代器方式，内存高效 ----
# 文件对象是可迭代的，for 循环内部通过迭代器协议每次读取一行
# 不会一次性将整个文件加载到内存，适合处理 GB 级别的大文件
print("\n=== 方式一：for line in f（迭代器，推荐） ===")
line_count = 0
with open("/tmp/big_file.txt", "r", encoding="utf-8") as f:
    for line in f:
        # 每次循环只占用一行数据的内存，处理完即释放
        line_count += 1
        # 只打印前 3 行作为预览，避免输出刷屏
        if line_count <= 3:
            # line.strip() 去除行末的换行符 \n 和两端空白字符
            print(f"  {line.strip()}")
print(f"  ... 共 {line_count} 行")

# ---- 方式二：f.readline() — 手动逐行读取 ----
# 每次调用 readline() 读取一行（包括行末换行符）
# 到达文件末尾时返回空字符串 ""
# 需要手动判断结束条件，容易出现死循环，不如 for 循环简洁
print("\n=== 方式二：f.readline() ===")
with open("/tmp/big_file.txt", "r", encoding="utf-8") as f:
    for _ in range(3):
        line = f.readline()
        if not line:  # 读取到文件末尾时 readline() 返回空字符串
            break
        print(f"  {line.strip()}")

# ---- 方式三（不推荐）：readlines() — 全部加载到内存 ----
# f.readlines() 会一次性读取所有行，返回一个列表，每行作为列表的一个元素
# 如果文件有 10GB，这行代码就会消耗 10GB+ 的内存！
# 仅适合小文件或确定文件很小时使用
print("\n=== 方式三：f.readlines()（不推荐，演示对比） ===")
with open("/tmp/big_file.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()  # 全部加载到列表！大文件慎用
print(f"  列表长度：{len(lines)} 行")
print(f"  第 500 行：{lines[499].strip()}")  # 列表索引从 0 开始，所以第 500 行是索引 499

# 清理临时文件
import os
os.remove("/tmp/big_file.txt")
print("\n临时文件已清理。")

## 5. JSON —— 通用数据交换格式

JSON（JavaScript Object Notation）是目前最广泛使用的**跨语言数据格式**。
几乎所有编程语言都能读写 JSON。

### 为什么 JSON 是"通用语言"？

想象你只会中文，朋友只会法语。你们之间的沟通障碍就是"语言不同"。
JSON 就像**世界语**——一种中立的、所有人都能理解的格式。

### Python 的 json 模块

| 方法 | 用途 | 类比 |
|------|------|------|
| `json.dumps(obj)` | Python 对象 → JSON **字符串** | 把大象装进文字描述 |
| `json.loads(s)` | JSON **字符串** → Python 对象 | 从文字描述还原大象 |
| `json.dump(obj, f)` | Python 对象 → 直接**写入文件** | 把大象装进集装箱 |
| `json.load(f)` | 从**文件**读取 → Python 对象 | 从集装箱取出大象 |

### 序列化与反序列化的内部机制

**序列化 (serialization)**：`json.dumps()` 将 Python 对象转为 JSON 字符串的过程：

```python
# 内部简化流程
data = {"name": "张三", "age": 20}

# 1. Python dict 递归遍历所有键值对
# 2. 每个 Python 类型按映射表转为 JSON 类型
# 3. 字符串会被编码为 UTF-8（默认 ensure_ascii=True 时，非 ASCII 转 \uXXXX）
# 4. 最终拼接为合法 JSON 字符串：{"name": "张三", "age": 20}
```

**反序列化 (deserialization)**：`json.loads()` 将 JSON 字符串转为 Python 对象，步骤正好相反：
1. 词法分析：JSON 解析器将字符串拆解为 token（`{`、`}`、`:`、字符串、数字等）
2. 语法分析：根据 JSON 语法规则构建树状结构
3. 类型映射：每个 JSON 值按映射表转为对应的 Python 类型
4. 返回 Python 的 dict/list/str/int/float/bool/None 组成的对象

### Python 与 JSON 类型映射

| Python 类型 | JSON 类型 | 说明 | 注意事项 |
|------------|----------|------|---------|
| `dict` | `{}` object | 键必须为字符串 | Python 的 int 键会被转成 str |
| `list` / `tuple` | `[]` array | tuple 序列化为 JSON 数组 | 反序列化后变成 list，不是 tuple |
| `str` | `""` string | 必须是 Unicode 字符串 | 默认 `ensure_ascii=True` 转义非 ASCII |
| `int` / `float` | number | int 范围不限 | float 的 NaN/Infinity 不合法 ❌ |
| `True` / `False` | `true` / `false` | ⚠️ 大小写不同！ | 反序列化后变回 Python 的 True/False |
| `None` | `null` | ⚠️ 注意转换 | 反序列化后变回 Python 的 None |
| `set` / `bytes` / `datetime` | ❌ 不支持 | 需要自定义编码器 | 需继承 `json.JSONEncoder` |

**类型映射的边角案例**：

```python
# tuple 会变成 list
json.loads(json.dumps((1, 2, 3)))  # 得到 [1, 2, 3] 而不是 (1, 2, 3)

# dict 的 int 键会被转为字符串
json.dumps({1: "a", 2: "b"})  # 得到 {"1": "a", "2": "b"}

# float 的特殊值无法序列化
json.dumps(float('nan'))  # ValueError: Out of range float values are not JSON compliant
```

### 编码细节：ensure_ascii 与中文

默认情况下 `ensure_ascii=True`，所有非 ASCII 字符（包括中文）会被转义为 `\uXXXX` 形式：

```python
json.dumps({"name": "张三"})
# 默认：{"name": "张三"}
# 设置 ensure_ascii=False：{"name": "张三"}
```

`ensure_ascii=False` 的好处是文件更可读，且体积略小。两种方式在 JSON 规范层面是**完全等价**的——任何标准 JSON 解析器都能正确解析。

### 常见误区

- **`dumps` 和 `dump` 分不清** —— 带 `s` 的是 **s**tring（字符串），不带 `s` 的是文件
- **JSON 键必须用双引号** —— `{'key': 1}` 不是合法 JSON，`{"key": 1}` 才是
- **`True` 变成 `true`** —— 从 JSON 读回后 `true` 变成 `True`，和 JS 不同
- **中文写入乱码** —— 设置 `ensure_ascii=False` 让中文保持可读
- **json.loads() 不是 eval()** —— 永远不要用 `eval()` 解析 JSON，有严重安全风险！`json.loads()` 是安全的，只解析合法 JSON

In [ ]:
# === JSON 序列化与反序列化 ===
import json  # json 模块：Python 内置的 JSON 编解码工具

# ---------- dumps / loads（字符串层面）----------
# JSON 的 "S" 代表 String——dumps 和 loads 在**字符串**层面操作，不涉及文件

data = {
    "name": "张三",
    "scores": [95, 88, 92],
    "passed": True,      # Python 的 bool 值，JSON 序列化后会变成 true
    "remark": None       # Python 的 None，JSON 序列化后会变成 null
}
# 注意：Python dict 的键可以是任意不可变类型，但 JSON 的键必须是双引号字符串

# ---- json.dumps()：Python 对象 → JSON 字符串 ----
# ensure_ascii=False：允许输出非 ASCII 字符（如中文），否则中文字符会被转义为 \uXXXX
# indent=2：美化输出，每层缩进 2 个空格，便于人眼阅读
json_str = json.dumps(data, ensure_ascii=False, indent=2)
print("=== dumps() 输出（JSON 字符串）===")
print(json_str)
print(f"类型：{type(json_str)}")  # <class 'str'>

# ---- json.loads()：JSON 字符串 → Python 对象 ----
# 将 JSON 格式的字符串反序列化为 Python 的 dict/list 等对象
# true → True, false → False, null → None（注意大小写变化）
restored = json.loads(json_str)
print("\n=== loads() 还原 ===")
print(f"还原后的 name：{restored['name']}")
print(f"类型：{type(restored)}")   # <class 'dict'>
print(f"True 还原为：{restored['passed']}")   # JSON true → Python True
print(f"None 还原为：{restored['remark']}")   # JSON null → Python None

# ---------- dump / load（文件层面）----------
# 不带 "s" 的 dump/load 直接操作文件对象，底层仍然调用 dumps/loads + f.write/f.read

# json.dump()：Python 对象 → 直接写入文件
# 内部等价于：f.write(json.dumps(data, ensure_ascii=False, indent=2))
with open("/tmp/demo.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

# json.load()：从文件读取 → Python 对象
# 内部等价于：json.loads(f.read())
with open("/tmp/demo.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)

print("\n=== dump / load 文件读写 ===")
print(loaded)

import os
os.remove("/tmp/demo.json")
print("\n临时文件已清理。")

## 6. CSV —— 表格数据格式

CSV（Comma-Separated Values）是最简单的**表格数据存储格式**。
Excel、数据库、各种数据分析工具都支持 CSV。

### 为什么用 csv 模块而不是手动 split？

```python
# 手动解析的坑：
line = '张三,90,"你好，世界",100'
# 用 split(',') 会错误分割引号内的逗号！
```

`csv` 模块自动处理了：
- 字段内包含逗号（用引号包裹）
- 字段内包含引号（用双引号转义，`""` 表示单个 `"`）
- 不同的换行符风格（Windows 的 `\r\n` vs Unix 的 `\n`）

### CSV 格式内部机制

CSV 格式虽然叫"逗号分隔值"，但实际上远比你想象的复杂：

**RFC 4180 标准核心规则**：
1. 每行代表一条记录，用换行符分隔（`\r\n`）
2. 字段之间用逗号分隔
3. 如果字段包含逗号、换行符或双引号，必须用双引号包裹
4. 字段内的双引号用两个双引号转义：`"He said ""Hello"""`
5. 最后一行的末尾可以有或没有换行符

**csv 模块的 Dialect（方言）**：
csv 模块支持不同风格的 CSV 变体，通过 `dialect` 参数指定：

| 方言 | 分隔符 | 引用符 | 换行符 | 适用场景 |
|------|--------|--------|--------|---------|
| `excel` (默认) | `,` | `"` | `\r\n` | Excel 导出的 CSV |
| `excel-tab` | `\t` | `"` | `\r\n` | 制表符分隔 (TSV) |
| `unix` | `,` | `"` | `\n` | Unix 风格 |

还可以自定义 Dialect：`csv.register_dialect('mycsv', delimiter='|', quoting=csv.QUOTE_ALL)`

### 数据序列化的类型问题

CSV 有一个根本性限制：**所有值都是字符串**。这与 JSON 不同，JSON 区分字符串、数字、布尔值和 null。

```python
csv.writerow(["123", "456"])    # 写入：123,456
csv.writerow([123, 456])        # 也会写入：123,456（writerow 会自动 str() 转换）
# 但是！！！读取回来时，所有值都是字符串："123" 和 "456"
# 数字需要手动 int() 或 float() 转换
```

这意味着 CSV 无法保留数据类型信息。如果需要精确的类型，需要：
1. 自己手动转换（`int(row["age"])`）
2. 使用专门的序列化格式（JSON、Parquet、Protocol Buffers）

### DictReader 和 DictWriter

比普通 reader/writer 更方便——直接用**列名访问**，就像操作字典：

| 方式 | 访问方式 | 适合场景 |
|------|---------|---------|
| `csv.reader` | `row[0]`、`row[1]` | 无表头或列顺序固定 |
| `csv.DictReader` | `row["姓名"]`、`row["分数"]` | **推荐**，可读性强 |

### Windows 坑：newline=""

在 **Windows** 上写入 CSV 文件时，必须指定 `newline=""`，否则会在每行末尾多一个空行。
这是因为 `csv` 模块自己处理换行，不需要 `open()` 再帮忙加一遍。

如果不设置 `newline=""`，在 Windows 上会发生：
- `csv.writer` 内部写入行时加 `\r\n`
- `open()` 在文本模式下看到 `\n`，又把它转成 `\r\n`
- 结果：`\r\r\n`，Excel 打开时每行之间多一个空行

### 编码问题：Excel 与 UTF-8

Excel 打开 UTF-8 编码的 CSV 时经常出现中文乱码，有两个解决方案：

| 方案 | 做法 | 优点 | 缺点 |
|------|------|------|------|
| UTF-8 BOM | `encoding="utf-8-sig"` | Excel 能正确识别 UTF-8 | 多 3 个字节前缀 |
| GBK 编码 | `encoding="gbk"` | Windows 中文系统兼容 | Linux 上无此编码 |

推荐使用 `utf-8-sig`，这样用 Python 写 CSV、用 Excel 双击打开都不会乱码。

### 常见误区

- **忘记 `newline=""`** —— Windows 上出现多余空行
- **编码问题** —— Excel 打开 CSV 乱码，可以加 `encoding="utf-8-sig"`（带 BOM）
- **手动 split 解析 CSV** —— 遇到引号内逗号就崩了
- **认为 CSV 能保存数据类型** —— 所有值都是字符串，数字需要手动转换
- **字段名中包含特殊字符** —— 如逗号、引号，DictWriter 写入时字段名也会被 CSV 规则处理
- **不同地区的 CSV 分隔符差异** —— 某些欧洲国家使用 `;` 作为分隔符（Excel 的区域设置），读取时需指定 `delimiter=';'`

In [ ]:
# === CSV 读写演示 ===
import csv  # csv 模块：Python 内置的 CSV 文件读写工具

# ---------- 写入 CSV ----------
# 数据准备：一个包含 dict 的列表，每个 dict 代表一条学生记录
students = [
    {"姓名": "张三", "语文": 95, "数学": 88, "英语": 92},
    {"姓名": "李四", "语文": 78, "数学": 94, "英语": 85},
    {"姓名": "王五", "语文": 88, "数学": 76, "英语": 90},
]

# newline=""：在 Windows 上必须指定，避免 csv 模块写入时出现多余空行
# 原因：csv 模块自己管理换行（writerow 内部会加 \r\n），open() 不需要再加一遍
# 如果不设置 newline=""，Windows 上每行末尾会多一个空行
with open("/tmp/grades.csv", "w", encoding="utf-8", newline="") as f:
    # fieldnames：指定列的顺序，DictWriter 按此顺序写入 CSV
    fieldnames = ["姓名", "语文", "数学", "英语"]
    # DictWriter：将字典写入 CSV，字典的键必须与 fieldnames 匹配
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    # writeheader()：写入表头行，即 fieldnames 列表中的字段名
    writer.writeheader()
    # writerows()：一次性写入多行数据（接受可迭代对象，如列表）
    writer.writerows(students)

print("CSV 写入完成。")

# ---------- 读取 CSV ----------
print("\n=== DictReader 读取 ===")
# DictReader：将 CSV 的每一行读取为一个 OrderedDict，键为表头字段名
# 比 csv.reader 更直观，可以直接用列名访问数据
with open("/tmp/grades.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)  # 自动将第一行作为字段名
    for row in reader:
        # 通过列名（表头中的字段）访问数据
        姓名 = row["姓名"]
        # CSV 中的所有值都是字符串，需要手动转换为数字类型
        总分 = int(row["语文"]) + int(row["数学"]) + int(row["英语"])
        print(f"  {姓名}：语文 {row['语文']}，数学 {row['数学']}，"
              f"英语 {row['英语']}，总分 {总分}")

# ---------- csv.reader (不使用表头) ----------
# csv.reader：将 CSV 的每一行读取为一个列表，不自动处理表头
print("\n=== reader 普通方式读取 ===")
with open("/tmp/grades.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        # 第一行（i==0）是表头，需要手动判断
        if i == 0:
            print(f"  表头：{row}")
        else:
            # 通过索引访问各列，可读性不如 DictReader
            print(f"  数据：{row}")

import os
os.remove("/tmp/grades.csv")
print("\n临时文件已清理。")

## 7. 异常处理 —— try/except/else/finally

### 为什么需要异常处理？

程序不可能永远一帆风顺：
- 用户输入了非法数据
- 要读取的文件不存在
- 网络连接断开
- 磁盘空间不足

异常处理就像**汽车的保险杠和安全气囊**——平时用不到，但关键时刻能救命。

### Python 的异常处理哲学：EAFP

Python 社区有一个核心设计哲学，来自 Python 之父 Guido van Rossum：

> **EAFP**（Easier to Ask for Forgiveness than Permission，先做再请求原谅）
>
> 相比"先检查再操作"（LBYL，Look Before You Leap），Python 更推崇"直接尝试，失败再处理"。

```python
# LBYL（C 风格，不推荐——存在竞态条件）
if os.path.exists("file.txt"):     # 检查时文件存在
    with open("file.txt") as f:    # 打开时文件可能已被删除！
        data = f.read()

# EAFP（Python 风格，推荐——直接做，出错了再说）
try:
    with open("file.txt") as f:
        data = f.read()
except FileNotFoundError:
    data = ""
```

**EAFP 的核心优势**：LBYL 存在竞态条件（检查到使用之间状态可能改变），而 EAFP 在出错时处理，既安全又简洁。

### 完整结构

```python
try:
    # 可能会出错的代码
except SomeException:
    # 出错了，怎么处理
else:
    # 没有出错时执行（可选）
finally:
    # 无论是否出错都会执行（可选）
```

### 执行流程

```
         ┌─────────┐
         │   try   │
         └────┬────┘
              │
    ┌─────────┼─────────┐
    │ 有异常  │ 无异常  │
    ↓         │         ↓
 ┌──────┐    │    ┌──────┐
 │except│    │    │ else │
 └──────┘    │    └──────┘
    │        │         │
    └────────┼─────────┘
             ↓
        ┌─────────┐
        │ finally │  ← 总是执行
        └─────────┘
```

### else 有什么用？（容易被忽略的重要角色）

很多初学者不理解为什么需要 `else`，认为可以把它放在 try 块里。但关键区别在于：

```python
# 错误：try 块太大，意外捕获了不该捕获的异常
try:
    result = risky_operation()
    save_to_database(result)     # 如果这行出错，也会被 except 吞没
except ValueError:
    print("risky_operation 失败")
    # 但 save_to_database 的错误被静默忽略了！！！

# 正确：else 让异常边界清晰
try:
    result = risky_operation()   # 只在这里捕获 ValueError
except ValueError:
    print("risky_operation 失败")
else:
    save_to_database(result)     # 只有 risky_operation 成功时才执行，其异常不会被错误捕获
```

**核心原则**：try 块只放**预期会抛出特定异常**的代码，其他代码放在 else 中。

### 异常传播机制（深入理解栈回溯）

当异常发生时，Python 解释器会：

1. **展开调用栈**：在当前函数查找匹配的 `except`，没找到则向调用者传播
2. **逐层回溯**：一直传播到 `__main__` 模块，如果还没有被捕获，程序终止并打印 **traceback**
3. **traceback 的结构**：
   ```
   Traceback (most recent call last):
     File "a.py", line 3, in inner    ← 最内层调用
       return 1 / 0                    ← 错误发生的位置
     File "a.py", line 6, in outer    ← 外层调用
       return inner()                 ← 调用链
     File "a.py", line 9, in <module> ← 顶层模块
       outer()                        ← 程序入口
   ZeroDivisionError: division by zero
   ```

### OS 层面：异常与信号

Python 的异常机制建立在**操作系统信号（Signal）**之上吗？并不完全是。Python 的异常是语言层面的控制流，不依赖 OS 信号（但 `KeyboardInterrupt` 确实是由 OS 的 SIGINT 信号触发的）。

### 常见误区

- **`else` 放在 `except` 后面，不是 `finally` 后面**
- **`finally` 里不要用 `return`** —— 它会覆盖 try 里的 return 值
- **空 `except:` 捕获所有异常** —— 包括 `KeyboardInterrupt`（按 Ctrl+C 都停不了）
- **异常吞没** —— `except: pass` 是最糟糕的写法，错误被静默忽略
- **`raise` 和 `raise e` 的微妙区别** —— 裸 `raise` 保留原始 traceback，`raise e` 会创建新的 traceback（Python 3 中差异已很小，但推荐裸 `raise`）

In [ ]:
# === 异常处理流程演示 ===

def divide(a, b):
    """演示 try/except/else/finally 的完整流程"""
    print(f"\\n调用 divide({a}, {b})")
    try:
        # try 块：放置可能引发异常的代码
        # 如果 b == 0，Python 运行时会在 CPU 层面触发除零异常
        # ZeroDivisionError 被抛出，try 块剩余的代码会被跳过
        result = a / b
    except ZeroDivisionError:
        # except 块：当 try 块中抛出 ZeroDivisionError 时执行
        # **只有匹配的异常类型**才会被捕获，其他异常继续向外传播
        # 执行完 except 后，跳过 else 块，直接进入 finally 块
        print("  [except] 捕获到 ZeroDivisionError：除数不能为零！")
        # return 语句：函数在这里返回，但 finally 仍然会在 return 之前执行
        return None
    else:
        # else 块：**只有当 try 块没有抛出任何异常时**才会执行
        # 如果 try 块中使用了 return/break/continue，else 也会执行
        # 注意：else 在 except 之后，finally 之前
        print("  [else] 没有异常，计算结果为：", result)
        return result
    finally:
        # finally 块：**无论是否发生异常都会执行**
        # 即使 try 或 except 中有 return，finally 也会在 return 之前运行
        # 通常用于资源清理（关闭文件、释放锁、关闭数据库连接）
        print("  [finally] 这段代码始终执行。")
    # 注意：如果 except 中有 return，函数不会走到这里

# 正常情况：b=2，不会抛出异常
# 执行顺序：try → else（无异常）→ finally
divide(10, 2)

# 异常情况：b=0，抛出 ZeroDivisionError
# 执行顺序：try（抛出异常）→ except（捕获处理）→ finally
divide(10, 0)

# 看看 finally 在 return 后的行为
def test_finally():
    """演示 finally 在 return 之后仍然执行"""
    try:
        # try 块执行 return 语句，函数本应在此返回
        return "from try"
    finally:
        # 但 finally 块会在 return **之前**被执行！
        # 这是因为 Python 解释器在执行 return 前会先检查是否有 finally
        # 注意：不要在 finally 里写 return，它会覆盖 try 的 return 值
        print("  finally 在 return 之后依然执行！")
        # 如果 finally 里有 return，函数会返回 finally 的 return 值
        # 而不是 try 的 return 值——这是常见的陷阱

print("\\n=== finally 的执行时机 ===")
result = test_finally()
print(f"  函数返回：{result}")

## 8. 常见异常类型

Python 内置了丰富的异常类型，形成**继承体系**：

```
BaseException
 ├── SystemExit         ←  sys.exit()
 ├── KeyboardInterrupt  ←  Ctrl+C
 └── Exception          ←  所有常规异常的基类
      ├── ValueError        ←  值不合法（如 int("abc")）
      ├── TypeError         ←  类型不匹配（如 1 + "a"）
      ├── KeyError          ←  字典键不存在
      ├── IndexError        ←  列表索引越界
      ├── FileNotFoundError ←  文件不存在（Python 3 新增）
      ├── ZeroDivisionError ←  除以零
      ├── AttributeError    ←  对象没有该属性
      ├── ImportError       ←  导入模块失败
      ├── OSError           ←  操作系统错误（文件权限、磁盘满等）
      │    └── FileNotFoundError, PermissionError, ...
      ├── RuntimeError      ←  运行时通用错误
      │    └── RecursionError ←  递归过深（栈溢出）
      └── StopIteration     ←  迭代器耗尽（for 循环内部使用）
```

### 为什么继承体系很重要？

异常继承体系不仅仅是分类——它直接决定了 `except` 的捕获行为：

```python
try:
    open("不存在的文件.txt")
except OSError:  # OSError 是 FileNotFoundError 的父类
    pass         # ✅ 这样也能捕获到 FileNotFoundError

try:
    1 / 0
except ArithmeticError:  # ArithmeticError 是 ZeroDivisionError 的父类
    pass                 # ✅ 也能捕获
```

**捕获的匹配规则**：`except SomeException` 会捕获 `SomeException` **及其所有子类**。这意味着：
- `except Exception` 捕获所有常规异常（通常这是最宽泛的安全捕获边界）
- `except BaseException` 甚至捕获 `KeyboardInterrupt` 和 `SystemExit`（几乎从不应该这么写）
- `except FileNotFoundError` 只捕获文件不存在的异常，不影响 `PermissionError`

### 文件操作相关的异常全家桶

| 异常 | 触发场景 | 典型原因 |
|------|---------|---------|
| `FileNotFoundError` | `open()` 不存在的文件（`"r"` 模式） | 路径写错、文件被删除 |
| `PermissionError` | 无权限读取/写入文件 | 文件只读、没有目录权限 |
| `IsADirectoryError` | 对目录执行 `open()` 读操作 | 路径指向目录而非文件 |
| `NotADirectoryError` | 路径中某部分不是目录 | 路径拼接错误 |
| `FileExistsError` | `"x"` 模式下文件已存在 | 锁文件/临时文件冲突 |
| `BlockingIOError` | 非阻塞模式下操作会阻塞 | 管道/网络文件描述符 |
| `OSError` | 其他操作系统错误 | 磁盘满、inode 耗尽 |

### 什么时候应该捕获什么级别的异常？

```python
# 具体异常 → 精确处理（推荐）
try:
    with open("data.csv") as f:
        ...
except FileNotFoundError:
    # 精确处理：文件不存在，创建默认文件
    init_default_data()

# 父类异常 → 批量处理
try:
    with open("data.csv") as f:
        ...
except OSError as e:
    # 统一处理所有 OS 层面的文件错误
    print(f"文件操作失败（errno={e.errno}）：{e.strerror}")

# 异常大杂烩 → 捕获所有常规异常
try:
    result = complex_operation()
except Exception as e:
    # 兜底：记录日志，防止程序崩溃
    logging.exception("操作失败")
    raise  # 记录后重新抛出，不让异常被吞没
```

### 关键原则

- **永远捕获 `Exception` 而不是 `BaseException`** —— 否则连 Ctrl+C 都被抓了
- **精确捕获** —— 能写 `except FileNotFoundError` 就不要写 `except Exception`
- **异常也是对象** —— 可以用 `as e` 获取异常实例，打印详细信息
- **`OSError` 有 `errno` 和 `strerror` 属性** —— 可以获取操作系统错误码和描述信息
- **Python 3 将 `IOError` 合并为 `OSError` 的别名** —— 现在写 `except OSError` 即可覆盖所有 IO 错误

In [ ]:
# === 常见异常类型演示 ===

def demo_exception(exc_type, code):
    """辅助函数：执行可能抛出异常的代码，捕获后打印异常类型和消息
    参数：
        exc_type: 异常类型（这里仅用于文档标注，实际未使用）
        code: 要执行的 Python 代码字符串
    """
    try:
        exec(code)  # exec() 动态执行字符串形式的 Python 代码
    except Exception as e:
        # type(e).__name__ 获取异常类的名称（如 "ValueError"）
        # str(e) 获取异常的描述信息
        print(f"  {type(e).__name__}: {e}")

print("=== 常见异常类型 ===\\n")

# ---- ValueError：值不合法 ----
# int() 期望字符串中是一个合法的整数表示
# "abc" 不是数字字符串，int() 无法转换，抛出 ValueError
demo_exception(ValueError, 'int("abc")')
# ValueError: invalid literal for int() with base 10: 'abc'

# ---- TypeError：类型不匹配 ----
# int 和 str 是两种不同的类型，+ 运算符不知道如何将 1 和 "hello" 相加
# Python 是强类型语言，不会隐式转换不同类型
demo_exception(TypeError, '1 + "hello"')
# TypeError: unsupported operand type(s) for +: 'int' and 'str'

# ---- KeyError：字典键不存在 ----
# 字典 {"a": 1} 中没有键 "b"，用 [] 访问不存在的键会抛出 KeyError
# 可以使用 .get() 方法避免此异常
demo_exception(KeyError, '{"a": 1}["b"]')
# KeyError: 'b'

# ---- IndexError：列表索引越界 ----
# 列表 [1, 2, 3] 只有 3 个元素（索引 0,1,2），索引 100 超出范围
demo_exception(IndexError, '[1, 2, 3][100]')
# IndexError: list index out of range

# ---- FileNotFoundError：文件不存在 ----
# open() 在 "r" 模式下如果文件不存在，操作系统返回 ENOENT 错误码
# Python 将其封装为 FileNotFoundError（Python 3 新增，之前是 IOError）
demo_exception(FileNotFoundError, 'open("不存在的文件.txt")')
# FileNotFoundError

# ---- ZeroDivisionError：除以零 ----
# CPU 层面不支持整数除以零，Python 抛出 ZeroDivisionError
demo_exception(ZeroDivisionError, '1 / 0')
# ZeroDivisionError: division by zero

# ---- AttributeError：对象没有该属性 ----
# 字符串没有 'upper' 属性（应该是 'upper()' 方法调用写成了属性访问）
demo_exception(AttributeError, 'NoneType没有这个方法'.upper')
# AttributeError: 'str' object has no attribute 'upper'

print("\\n=== 多异常捕获（一个 except 抓多个）===")
# 同一个 except 可以捕获多个异常类型，用元组列出
try:
    x = int("abc")
except (ValueError, TypeError) as e:  # 同时捕获 ValueError 和 TypeError
    print(f"  捕获到：{type(e).__name__}: {e}")

print("\\n=== 获取异常完整信息 ===")
import traceback  # traceback 模块：获取完整的异常堆栈追踪信息

try:
    1 / 0
except ZeroDivisionError as e:
    print("  简短信息：", e)           # 异常的字符串描述
    print("  异常类型：", type(e).__name__)  # 异常类的名称
    print("  完整追踪：")
    traceback.print_exc()  # 打印完整的调用堆栈跟踪（包括文件路径和行号）

## 9. 异常处理最佳实践

### 原则一：捕获具体异常

```python
# 错误 —— 吞没了所有异常
try:
    result = risky_operation()
except:
    pass

# 正确 —— 只捕获预期内的异常
try:
    result = risky_operation()
except (ValueError, FileNotFoundError) as e:
    log.error(f"操作失败：{e}")
    raise  # 或者做合理处理
```

### 原则二：不要吞没异常（Don't Swallow Errors）

用一个空的 `except: pass` 是**最危险**的写法：
- 你永远不知道程序出了什么错
- 调试时毫无线索
- 可能导致数据损坏而不自知

### 原则三：使用自定义异常

当你的程序有特定的错误场景时，定义自己的异常类：

```python
class GradeError(Exception):
    """成绩相关的异常基类"""
    pass

class NegativeGradeError(GradeError):
    """负数成绩异常"""
    pass

class ExcessiveGradeError(GradeError):
    """成绩超过上限异常"""
    pass
```

自定义异常的好处：
1. **语义明确** —— 一看就知道是什么场景出的错
2. **层次清晰** —— 可以用 `except GradeError` 捕获所有成绩相关异常
3. **便于维护** —— 修改异常处理时不会影响其他模块

### 原则四：EAFP 优于 LBYL

Python 推崇 **EAFP**（Easier to Ask for Forgiveness than Permission，先做再请求原谅）
而不是 **LBYL**（Look Before You Leap，三思而后行）：

```python
# LBYL（C 风格，不推荐）
if os.path.exists("file.txt"):
    with open("file.txt") as f:
        data = f.read()

# EAFP（Python 风格，推荐）
try:
    with open("file.txt") as f:
        data = f.read()
except FileNotFoundError:
    data = ""
```

因为：检查存在 ≠ 可读（权限问题、竞态条件）

### 深入：自定义异常的设计模式

#### 什么时候需要自定义异常？

当你写一个模块或库时，如果满足以下任一条件，就应该自定义异常：

1. **你的模块可能抛出多种不同类型的错误** —— 让调用者可以分别捕获
2. **你的错误需要携带额外信息** —— 如错误码、非法值、建议操作
3. **你希望调用者 `except` 你的异常而不是 Python 内置异常** —— 解耦

#### 异常继承体系设计

```python
# 优秀的设计：两层继承体系
class DatabaseError(Exception):
    """数据库操作异常基类"""
    pass

class ConnectionError(DatabaseError):
    """数据库连接异常"""
    def __init__(self, host, port, timeout=30):
        self.host = host
        self.port = port
        self.timeout = timeout
        super().__init__(f"无法连接到 {host}:{port}（超时 {timeout}s）")

class QueryError(DatabaseError):
    """数据库查询异常"""
    def __init__(self, sql, original_error):
        self.sql = sql
        self.original_error = original_error
        super().__init__(f"查询失败：{sql} — {original_error}")

# 调用者可以按需选择捕获精度
try:
    db.query("SELECT * FROM users")
except ConnectionError:
    # 只处理连接问题
    retry()
except DatabaseError:
    # 处理所有数据库异常
    log.error("数据库操作失败")
```

#### 从什么类继承？

| 场景 | 推荐基类 | 原因 |
|------|---------|------|
| 通用业务异常 | `Exception` | 最通用的常规异常基类 |
| 模块级异常 | 自定义模块异常基类 | 从 `Exception` 继承，子异常继承此基类 |
| 与内置异常语义相同 | 对应的内置异常 | 如从 `ValueError`、`TypeError` 继承 |
| OS 相关错误 | `OSError` | 如文件系统、网络相关异常 |

**从内置异常继承的例子**：

```python
class InvalidAgeError(ValueError):
    """年龄不合法：语义上和 ValueError 一致，也是"值不合法""""
    pass

class MissingFieldError(KeyError):
    """缺少字段：语义上和 KeyError 一致"""
    pass
```

这样调用者可以 `except ValueError` 同时捕获 Python 内置的 `ValueError` 和你的 `InvalidAgeError`。

#### 异常应该包含什么信息？

一个好的异常应该包含：

```python
class FileFormatError(Exception):
    """文件格式错误，携带文件名、行号和具体问题"""
    def __init__(self, filename, line_number, detail):
        self.filename = filename
        self.line_number = line_number
        self.detail = detail
        # 错误消息应该让调用者一眼看出问题所在
        super().__init__(f"{filename}:{line_number} — {detail}")

# 使用
raise FileFormatError("config.json", 10, "缺少必要的 `version` 字段")
```

### 异常链（Exception Chaining）

Python 3 支持异常链，让你在重新抛出异常时保留原始异常的上下文：

```python
def read_config(path):
    try:
        with open(path) as f:
            return parse(f.read())
    except OSError as e:
        # raise ... from ... 保留原始异常链
        raise ConfigError(f"读取配置文件失败：{path}") from e

# 调用时可以看到完整链路：
# ConfigError: 读取配置文件失败：/etc/app/config.json
#   —— 由以下异常引起 ——
# FileNotFoundError: [Errno 2] No such file or directory: '/etc/app/config.json'
```

没有 `from` 的话，异常链会断开，调试时无法追溯到根因。**重新抛出被包装的异常时，请始终使用 `from`**。

### 总结：异常处理的黄金法则

| 要做 | 不要做 |
|------|--------|
| 捕获具体的异常类型 | 用空的 `except:` |
| 用 `as e` 获取异常信息 | 用 `except Exception` 替代具体异常 |
| 用 `else` 缩小 try 块范围 | 在 `finally` 中 `return` |
| 自定义异常让语义清晰 | 吞没异常（`except: pass`） |
| 用 `raise` 重新抛出（保留 traceback） | 用 `raise e` 重新抛出 |
| 用 `raise X from e` 链式包装异常 | 赋值异常给变量后忘记处理 |

In [ ]:
# === 异常处理最佳实践 ===
import json

# ---------- 原则：自定义异常 ----------
# 自定义异常让错误语义更清晰，调用者可以根据异常类型精确处理
# 所有自定义异常都应该继承自 Exception（而不是 BaseException）

class GradeError(Exception):
    """成绩相关的异常基类"""
    pass

class NegativeGradeError(GradeError):
    """负数成绩异常：当成绩为负数时抛出"""
    pass

class ExcessiveGradeError(GradeError):
    """成绩超过上限异常：当成绩超过 100 时抛出"""
    pass

def validate_score(score):
    """验证成绩是否合法
    参数 score：待验证的成绩数值
    返回值：合法的成绩（原样返回）
    异常：NegativeGradeError（负数）/ ExcessiveGradeError（超限）
    """
    if score < 0:
        # raise 语句：手动抛出异常，中断当前函数执行
        raise NegativeGradeError(f"成绩不能为负数，收到：{score}")
    if score > 100:
        raise ExcessiveGradeError(f"成绩不能超过 100，收到：{score}")
    return score

# ---------- 测试自定义异常 ----------
print("=== 自定义异常 ===")
for val in [95, -5, 150]:
    try:
        validate_score(val)
        print(f"  成绩 {val}：合法")
    except NegativeGradeError as e:
        # 精确捕获 NegativeGradeError，处理负数的逻辑
        print(f"  负数错误：{e}")
    except ExcessiveGradeError as e:
        # 精确捕获 ExcessiveGradeError，处理超限的逻辑
        print(f"  超限错误：{e}")
    except GradeError as e:
        # 基类捕获：兜底处理所有其他成绩相关异常
        # 注意：由于 NegativeGradeError 和 ExcessiveGradeError 继承自 GradeError
        # 如果上面的精确捕获没匹配到，这里才会被触发
        print(f"  其他成绩错误：{e}")

# 用基类一次性捕获多个子类
print("\\n=== 用基类 GradeError 捕获 ===")
try:
    validate_score(-1)
    # validate_score(200) 不会执行，因为上一行已经抛出异常
    # 这就是 EAFP：先做，有问题再处理
    validate_score(200)
except GradeError as e:
    # 一次捕获所有 GradeError 子类（多态）
    print(f"  捕获到成绩异常：{type(e).__name__}: {e}")

# ---------- EAFP 风格 ----------
# EAFP = Easier to Ask for Forgiveness than Permission
# 中文：先做再请求原谅（不如先斩后奏）
# 与 LBYL（Look Before You Leap，三思而后行）相对
# Python 推崇 EAFP，因为"检查存在"不等于"可读"（有权限问题、竞态条件等）
print("\\n=== EAFP 风格（先做再处理异常） ===")

def read_config(path):
    """EAFP 风格：直接尝试打开，失败再处理
    不先检查文件是否存在（LBYL），而是直接 try 操作
    """
    try:
        # 直接尝试打开和读取 JSON 文件
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        # 如果文件不存在→FileNotFoundError，返回默认空配置
        print(f"  配置文件 {path} 不存在，返回空配置")
        return {}
    except json.JSONDecodeError as e:
        # 如果文件内容不是合法 JSON→JSONDecodeError，也返回默认空配置
        print(f"  配置文件格式错误：{e}")
        return {}
    # 如果 open() 成功且 json.load() 成功，函数正常返回配置数据
    # 不需要 else 块——try 块中的 return 已经处理了成功路径

config = read_config("/tmp/nonexistent_config.json")
print(f"  读取结果：{config}")

## 10. 综合实战：成绩管理系统

将文件 I/O 和异常处理结合起来，构建一个完整的**成绩管理系统**。

### 需求

1. **添加学生成绩** —— 输入姓名和成绩，保存到 CSV 文件
2. **查看全部成绩** —— 从 CSV 文件读取并显示
3. **统计汇总** —— 计算平均分、最高分、最低分
4. **数据校验** —— 成绩必须在 0~100 之间，姓名不能为空
5. **文件异常处理** —— 文件不存在、格式错误等都能优雅处理

### 设计思路

- 使用 `csv.DictReader` / `csv.DictWriter` 读写数据
- 自定义异常类 `GradeError`、`NegativeGradeError`、`ExcessiveGradeError`
- 使用 `with` 语句确保文件正确关闭
- 所有可能的异常都要处理，不让程序崩溃
- 使用 `json` 格式导出统计报告

### 架构概要：从分层角度看这段代码

这段实战代码展示了 Python 文件处理中的**三层架构**：

```
用户接口层
  │   main()：组织测试数据、调用各函数、输出结果
  │
  ├── 业务逻辑层
  │   ├── validate_score()：成绩校验（EAFP 风格）
  │   ├── validate_name()：姓名校验
  │   └── generate_report()：统计计算
  │
  ├── 数据存取层
  │   ├── add_student()：读写 CSV 文件（with + csv.DictWriter）
  │   └── show_all()：读取 CSV 文件（with + csv.DictReader）
  │
  └── 异常体系层
      ├── GradeError（Exception）：基础异常类
      ├── NegativeGradeError（GradeError）：负数异常
      └── ExcessiveGradeError（GradeError）：超限异常
```

### 设计决策背后的原理

| 决策 | 为什么这样设计？ | 关联知识点 |
|------|----------------|-----------|
| 自定义异常继承自 `Exception` | 让调用者用 `except GradeError` 统一捕获 | 异常继承体系、精确捕获 |
| 数据校验直接 `raise` | EAFP 风格：不做预检查，出问题再抛异常 | EAFP vs LBYL |
| `with open(...) as f` | 保证 `__exit__` 自动调用，不会 fd 泄漏 | 上下文管理器协议 |
| `csv.DictReader` 而不是 `csv.reader` | 用列名访问数据，代码可读性更高 | CSV Dialect 机制 |
| 异常链使用裸 `raise` | 保留原始 traceback，方便调试 | Stack unwinding |

### 学习目标

1. 理解文件 I/O 在实际项目中的应用
2. 掌握异常处理的完整流程（try/except/else/finally）
3. 学会自定义异常和精确捕获
4. 体会 EAFP 风格编程的优雅
5. **思考题**：如果成绩文件同时被两个程序写，会发生什么？（竞态条件，思考如何用 `"x"` 模式加锁）

请运行下面的代码，并尝试修改和扩展功能！

In [ ]:
# === 综合实战：成绩管理系统 ===
# 将文件 I/O、异常处理、JSON、CSV 等知识综合运用，构建一个完整的成绩管理系统

import csv      # csv 模块：读写 CSV 格式的成绩数据文件
import json     # json 模块：导出 JSON 格式的统计报告
import os       # os 模块：文件路径判断和删除

# ---------- 自定义异常 ----------
# 通过自定义异常类体系，让错误语义更清晰、捕获更精确

class GradeError(Exception):
    """成绩相关异常基类，捕获所有成绩相关异常的基类"""
    pass

class NegativeGradeError(GradeError):
    """负数成绩异常：成绩为负数时抛出"""
    pass

class ExcessiveGradeError(GradeError):
    """成绩超过上限异常：成绩超过 100 时抛出"""
    pass

# ---------- 数据校验 ----------
def validate_score(score):
    """验证成绩，返回浮点数
    步骤：①尝试转为 float ②检查负数 ③检查超限
    EAFP 风格：先转类型，转换失败再处理异常
    """
    try:
        # 尝试将输入转换为浮点数——可能收到字符串 "abc" 等非法值
        score = float(score)
    except (ValueError, TypeError):
        # ValueError：字符串不能转为数字（如 "abc"）
        # TypeError：类型完全不支持转换（如 None）
        # 抛出 GradeError（基类），上层可以用 except GradeError 统一捕获
        raise GradeError(f"成绩必须是数字，收到：{score!r}")
    if score < 0:
        raise NegativeGradeError(f"成绩不能为负数，收到：{score}")
    if score > 100:
        raise ExcessiveGradeError(f"成绩不能超过 100，收到：{score}")
    return score

def validate_name(name):
    """验证姓名，去除首尾空白后返回
    空姓名或纯空白姓名被视为非法
    """
    if not name or not name.strip():
        # not name：捕获 None 或空字符串
        # not name.strip()：捕获纯空白字符串如 "   "
        raise GradeError("姓名不能为空")
    return name.strip()

# ---------- 数据操作 ----------
# 定义数据文件和报告文件的路径（使用 /tmp 目录避免污染工作目录）
DATA_FILE = "/tmp/grade_system.csv"
REPORT_FILE = "/tmp/grade_report.json"

def add_student(name, score):
    """添加一条成绩记录到 CSV 文件
    流程：校验数据 → 读取已有记录 → 追加新记录 → 写回文件
    """
    # 先校验数据，校验不通过直接抛出异常（不碰文件）
    name = validate_name(name)
    score = validate_score(score)

    # 读取已有数据（如果文件存在）
    records = []
    try:
        # 尝试打开已有 CSV 文件读取现有记录
        with open(DATA_FILE, "r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                records.append(row)
    except FileNotFoundError:
        # EAFP 风格：文件不存在是正常情况，从空列表开始即可
        pass  # 文件不存在，从空开始
    except csv.Error as e:
        # CSV 格式错误（如混合了不同的分隔符）
        print(f"警告：数据文件格式异常，将重新创建：{e}")
        records = []

    # 添加新记录（将数据转为 dict 放入列表）
    records.append({"name": name, "score": str(score)})

    # 写回文件（覆盖写入，包含所有旧记录 + 新记录）
    # 用 "w" 模式：先清空再全部重写，保证文件完整性
    with open(DATA_FILE, "w", encoding="utf-8", newline="") as f:
        # fieldnames 指定列的顺序
        writer = csv.DictWriter(f, fieldnames=["name", "score"])
        writer.writeheader()   # 写入 CSV 表头：name,score
        writer.writerows(records)  # 写入所有数据行

    print(f"已添加：{name} -> {score} 分")

def show_all():
    """显示所有成绩记录
    从 CSV 文件读取并格式化打印
    返回记录列表供其他函数使用
    """
    try:
        # 尝试打开 CSV 文件并读取所有记录
        with open(DATA_FILE, "r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            # list(reader) 一次性将迭代器转为列表——这里数据量小，没问题
            records = list(reader)
    except FileNotFoundError:
        print("暂无数据（文件不存在）")
        return []
    except csv.Error as e:
        print(f"数据文件格式错误：{e}")
        return []

    if not records:
        print("暂无数据")
        return []

    # 格式化打印：姓名左对齐 8 字符宽度，成绩左对齐 8 字符宽度
    print(f"\\n{'姓名':<8} {'成绩':<8}")
    print("-" * 16)
    for row in records:
        print(f"{row['name']:<8} {row['score']:<8}")
    return records

def generate_report():
    """生成统计报告并导出 JSON 文件
    从 CSV 读取数据 → 计算统计指标 → 写入 JSON 文件
    """
    records = show_all()  # 复用 show_all() 获取数据
    if not records:
        return

    # 提取并转换成绩分数（CSV 中所有值都是字符串）
    scores = []
    for row in records:
        try:
            # 将字符串分数转为 float，可能遇到非法值
            scores.append(float(row["score"]))
        except (ValueError, KeyError) as e:
            # ValueError：分数不是合法数字
            # KeyError：数据行缺少 "score" 键
            print(f"跳过无效记录 {row}：{e}")

    if not scores:
        print("没有有效成绩数据")
        return

    # 计算统计数据
    report = {
        "总计人数": len(scores),
        "平均分": round(sum(scores) / len(scores), 2),  # 保留两位小数
        "最高分": max(scores),
        "最低分": min(scores),
        "成绩列表": scores
    }

    # 将统计报告导出为 JSON 文件
    with open(REPORT_FILE, "w", encoding="utf-8") as f:
        # ensure_ascii=False：保持中文可读
        # indent=2：美化输出
        json.dump(report, f, ensure_ascii=False, indent=2)

    print(f"\\n=== 统计报告 ===")
    print(f"  总计人数：{report['总计人数']}")
    print(f"  平均分：{report['平均分']}")
    print(f"  最高分：{report['最高分']}")
    print(f"  最低分：{report['最低分']}")
    print(f"\\n报告已导出到：{REPORT_FILE}")

# ---------- 主流程 ----------
def main():
    """成绩管理系统主入口
    整合所有功能：添加测试数据 → 显示记录 → 生成报告 → 清理文件
    """
    print("=" * 40)
    print("      成绩管理系统")
    print("=" * 40)

    # 准备测试数据：包含合法数据和非法数据（负数、超限、空姓名）
    test_data = [
        ("张三", 95),
        ("李四", 78),
        ("王五", 88),
        ("赵六", -5),    # 非法：负数→触发 NegativeGradeError
        ("钱七", 150),   # 非法：超限→触发 ExcessiveGradeError
        ("", 90),        # 非法：空姓名→触发 GradeError
    ]

    # 逐个添加测试数据，每个都 try 捕获异常
    for name, score in test_data:
        try:
            add_student(name, score)
        except NegativeGradeError as e:
            # 精确捕获负数成绩异常
            print(f"  [错误] {e}")
        except ExcessiveGradeError as e:
            # 精确捕获超限成绩异常
            print(f"  [错误] {e}")
        except GradeError as e:
            # 基类兜底：捕获其他所有成绩相关异常
            print(f"  [错误] {e}")

    # 显示所有成绩记录
    show_all()

    # 生成并导出统计报告
    generate_report()

    # 清理临时文件
    for f in [DATA_FILE, REPORT_FILE]:
        if os.path.exists(f):
            os.remove(f)
    print(f"\\n临时文件已清理。")

# 程序入口：仅当直接运行此脚本时才执行 main()
main()
print("\\n成绩管理系统演示完成！")